### Lab 1.3: Multi-Class Linear Classifier

In this lab you will explore multi-class classification and evaluate model generalization using a [dataset for heart disease prediction from the UCI ML repository](https://archive.ics.uci.edu/dataset/45/heart+disease).

In [1]:
!pip install ucimlrepo

This ``ucimlrepo`` package provides a nice interface for accessing their datasets.

In [2]:
from ucimlrepo import fetch_ucirepo 

# fetch dataset 
heart_disease = fetch_ucirepo(id=45) 
  
# data (as pandas dataframes) 
X = heart_disease.data.features 
y = heart_disease.data.targets 
  
# variable information 
heart_disease.variables


,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,None,years,no
1,sex,Feature,Categorical,Sex,None,None,no
2,cp,Feature,Categorical,None,None,None,no
3,trestbps,Feature,Integer,None,resting blood pressure (on admission to the ho...,mm Hg,no
4,chol,Feature,Integer,None,serum cholestoral,mg/dl,no
5,fbs,Feature,Categorical,None,fasting blood sugar > 120 mg/dl,None,no
6,restecg,Feature,Categorical,None,None,None,no
7,thalach,Feature,Integer,None,maximum heart rate achieved,None,no
8,exang,Feature,Categorical,None,exercise induced angina,None,no
9,oldpeak,Feature,Integer,None,ST depression induced by exercise relative to ...,None,no


Here I remove the missing values from the features and labels.

In [3]:
bad = X.isna().any(axis=1)
X = X[~bad]
y = y[~bad]

Finally I convert the DataFrames to numpy arrays.

In [4]:
X = X.values
y = y.values.flatten()

The classification target is a number from 0-4 indicating the severity of heart disease.  Let's try fitting a linear model.

In [5]:
import sklearn

In [6]:
model = sklearn.linear_model.LogisticRegression().fit(X,y)

c:\Users\Logan\anaconda3\envs\csc487_env\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
model.score(X,y)

0.6094276094276094

### Exercises

1. Compute the $\mathbf{z}$ values for the classifier manually, i.e. compute

$$\mathbf{W}\mathbf{X}+\mathbf{b}.$$

*Hints*: 
- Use `.shape` to get the shape of a Numpy matrix.
- ``@`` is the matrix multiplication operator in Numpy
- The actual computation will be a little different from what is written above.  You will need to use a matrix transpose which is `.T` in Numpy.

In [8]:
print(f'Shape of X is {X.shape}')
print(f'Shape of y is {y.shape}')

W = model.coef_
b = model.intercept_
b = b.reshape(5,1)  # reshape to matrix so b can broadcast to (5, 297) when adding to W @ X.T

print(f'Shape of W is {W.shape}')
print(f'Shape of b is {b.shape}')

z = W @ X.T + b
print(f'Shape of z is {z.shape}')

Shape of X is (297, 13)
Shape of y is (297,)
Shape of W is (5, 13)
Shape of b is (5, 1)
Shape of z is (5, 297)


Print out the $\mathbf{z}$ values for the first example in the dataset and the first label.   Determine if the classifier is correctly classifying the first example in the dataset.

In [9]:
z_0 = z[:,0]
print(f'z values of first example:\n{z_0}')
print(f'argmax(z_0) is: {max(z_0)}, which is class 0')

z_0_model_prediction = model.predict(X[0,:].reshape(1,-1))
print(f'\nModel predicts the first example is: {z_0_model_prediction}')
print('Model is thus predicting the right class!')

z values of first example:
[ 1.02965543  0.44588516 -0.31956843 -0.3376924  -0.81827977]
argmax(z_0) is: 1.029655434884597, which is class 0

Model predicts the first example is: [0]
Model is thus predicting the right class!


2. Use ``sklearn.model_selection.train_test_split`` to split ``X`` and ``y`` into 90% train and 10% test splits.  Note that this should be done in a single call to ``train_test_split``.

*Note*: Pass ``random_state=1234`` to ``train_test_split`` to ensure you get the same result from random shuffling each time.


In [10]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size=0.10, random_state=1234)

Fit the model to the training split and calculate accuracy on the test split.  How does it compare to the previous accuracy value (when the model was trained and evaluated on the same data)?

In [11]:
model_split = sklearn.linear_model.LogisticRegression().fit(X_train,y_train)
model.score(X_test,y_test)

c:\Users\Logan\anaconda3\envs\csc487_env\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.5666666666666667

#### Model Comparison
It looks like the model trained on only 90% of the data had lower accuracy on the 10% of the data reserved for testing. This is only slightly lower than the 61% accuracy that the model showed when it was tested on the same data it was trained on, which is expected. This means that the model seems to be generalizing fairly well.

3. Run $k$-fold cross validation with $k=5$ and interpret the results (see `sklearn.model_selection.cross_val_score`).

In [12]:
linear_regression = sklearn.linear_model.LogisticRegression()
kfold_scores = sklearn.model_selection.cross_val_score(linear_regression, X, y, cv=5)

c:\Users\Logan\anaconda3\envs\csc487_env\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Logan\anaconda3\envs\csc487_env\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/p

In [13]:
print(kfold_scores)

[0.6        0.6        0.52542373 0.55932203 0.59322034]


#### Interpretation
Each of the five k-fold cross-validations also arrived at an accuracy of about 50-60%, so we can be more confident in that legitimacy of that accuracy metric for our model, i.e. the chance that our data reserved for the model validation is not mostly "easy" or "hard" data for the model, and is instead data representative of the entire dataset and thus indicative of true model performance.